# Vehicle Price Model KPI Report

## Executive Summary

This notebook rebuilds the model-output readout from the saved artifacts produced by `ML/Price_ML_Models.py` and `ML/Time_Series_Price.py`. It is intended to be rerun after model training so the KPI tables, segment checks, and forecast coverage update from the latest JSON/CSV outputs.


In [13]:
from __future__ import annotations

import json
from pathlib import Path

import joblib
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 30)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "ML":
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_DIR = PROJECT_ROOT / "MODELS_OUTPUT"
CURRENT_REPORT_PATH = OUTPUT_DIR / "model_report.json"
COHORT_REPORT_PATH = OUTPUT_DIR / "cohort_depreciation_model_report.json"
FORECAST_PATH = OUTPUT_DIR / "cohort_future_forecasts.csv"
TOP_MODEL_WEIGHT_ROWS = 30

def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing required report: {path}")
    return json.loads(path.read_text(encoding="utf-8"))

def load_forecast_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_csv(path)
    for column in ["observed_month_start", "observed_week_start", "forecast_date"]:
        if column in df.columns:
            df[column] = pd.to_datetime(df[column], errors="coerce")
    for column in [
        "forecast_month",
        "forecast_horizon_days",
        "predicted_depreciation_pct",
        "predicted_median_price",
        "observed_median_price",
        "unique_vins",
        "volume",
    ]:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")
    return df

current_report = load_json(CURRENT_REPORT_PATH)
cohort_report = load_json(COHORT_REPORT_PATH)
forecast_df = load_forecast_csv(FORECAST_PATH)

In [14]:
def money(value):
    if pd.isna(value):
        return "n/a"
    return f"${value:,.0f}"

def pct(value, digits=1):
    if pd.isna(value):
        return "n/a"
    return f"{value:.{digits}%}"

def horizon_months_from_target(target: str) -> int:
    try:
        token = str(target).split("_")[-1]
        if token.endswith("m"):
            return int(token.removesuffix("m"))
        if token.endswith("d"):
            return max(1, round(int(token.removesuffix("d")) / 30))
        return int(token)
    except Exception:
        return -1

best_model = current_report["recommended_model"]
best_metrics = current_report["models"][best_model]
cohort_models = cohort_report.get("models", {})
trained_cohort_models = {
    name: model for name, model in cohort_models.items()
    if not model.get("skipped") and model.get("metrics")
}
skipped_cohort_models = {
    name: model for name, model in cohort_models.items()
    if model.get("skipped")
}
cohort_summary_rows = []
for target, model in trained_cohort_models.items():
    metrics = model.get("metrics", {})
    cohort_summary_rows.append(
        {
            "target": target,
            "horizon_months": horizon_months_from_target(target),
            "future_price_mae": metrics.get("future_price_mae"),
            "future_price_wape": metrics.get("future_price_wape"),
            "skill_vs_naive": metrics.get("future_price_mae_skill_vs_naive"),
            "depreciation_mae_pct_points": metrics.get("depreciation_mae_pct_points"),
            "bias_dollars": metrics.get("future_price_bias"),
            "depreciation_r2": metrics.get("depreciation_r2"),
        }
    )
cohort_summary = pd.DataFrame(cohort_summary_rows)
best_cohort = (
    cohort_summary.sort_values(["skill_vs_naive", "future_price_wape"], ascending=[False, True]).iloc[0]
    if not cohort_summary.empty
    else None
)
max_forecast_date = cohort_report.get("max_forecast_date")
last_observed_month = cohort_report.get("last_observed_month") or cohort_report.get("last_observed_week")

summary_lines = [
    f"- **Current-price winner:** `{best_model}` with MAE {money(best_metrics.get('mae'))}, RMSE {money(best_metrics.get('rmse'))}, MAPE {pct(best_metrics.get('mape'))}, and R2 {best_metrics.get('r2', np.nan):.3f}.",
    f"- **Validation design:** {current_report['row_counts']['train_rows']:,} train rows and {current_report['row_counts']['test_rows']:,} test rows with VIN overlap of {current_report['split']['vin_overlap']}.",
    f"- **Cohort forecast coverage:** latest observed cohort month is {last_observed_month}; saved forecasts extend to {max_forecast_date}.",
]
if best_cohort is not None:
    summary_lines.append(
        "- **Best time-series KPI:** "
        f"`{best_cohort['target']}` has the strongest skill vs no-change at {pct(best_cohort['skill_vs_naive'])}, "
        f"with future-price MAE {money(best_cohort['future_price_mae'])} and WAPE {pct(best_cohort['future_price_wape'])}."
    )
    if best_cohort["skill_vs_naive"] <= 0:
        summary_lines.append(
            "- **Forecast guardrail:** the best validated cohort model does not beat the no-change baseline on MAE, so treat the recursive forecasts as directional scenarios until more history improves baseline skill."
        )
if skipped_cohort_models:
    skipped = ", ".join(f"`{target}` ({model.get('complete_rows', 0):,} complete rows)" for target, model in skipped_cohort_models.items())
    summary_lines.append(f"- **Long-horizon caveat:** direct models were skipped where labeled history is too sparse: {skipped}. Recursive forecast rows are marked separately in the forecast CSV.")

display(Markdown("\n".join(summary_lines)))

- **Current-price winner:** `Segmented_HighValue_LightGBM` with MAE $1,941, RMSE $6,295, MAPE 7.5%, and R2 0.937.
- **Validation design:** 2,678,670 train rows and 669,668 test rows with VIN overlap of 0.
- **Cohort forecast coverage:** latest observed cohort month is 2026-07-01; saved forecasts extend to 2031-07-01.
- **Best time-series KPI:** `target_depreciation_pct_1m` has the strongest skill vs no-change at -6.9%, with future-price MAE $1,968 and WAPE 5.1%.
- **Forecast guardrail:** the best validated cohort model does not beat the no-change baseline on MAE, so treat the recursive forecasts as directional scenarios until more history improves baseline skill.

## Context & Methods

The KPI set follows common regression and forecast-evaluation practice:

- **MAE** is the primary business KPI because it is measured in dollars and is easy to interpret for vehicle pricing decisions.
- **RMSE** is retained because it penalizes large misses more heavily than MAE.
- **MAPE/WAPE/RMSLE** provide relative-error views for mixed price bands; MAPE is useful here because prices are positive, but it should not be the only KPI.
- **R2** is a model-fit signal, not a business KPI by itself.
- **Bias and naive-baseline skill** matter for time-series forecasts because a forecast can be low-error on average while still consistently overpricing or underpricing.
- **Segment MAE** by price band, high-value status, make, and model year checks whether the headline model hides weak slices.
- **Feature weights/importance** are reported for interpretability. Linear readable models expose signed coefficients; tree and LightGBM-style models expose model-native importance rankings, which are directional diagnostics rather than causal claims.

Research scan:

- Used-car pricing papers support supervised tabular models with vehicle attributes, mileage, market context, and careful feature selection.
- Recent used-car pricing research also emphasizes uncertainty and trustworthiness, which makes segment checks, bias, and baseline comparisons important model-output KPIs.
- Global time-series forecasting papers support training one model across many related cohort series, especially when individual series are short or sparse.
- Forecast-accuracy references recommend testing against simple baselines and using scale-aware error measures; this is why the depreciation model reports no-change baseline skill and WAPE in addition to raw dollar error.


In [15]:
metric_contract = pd.DataFrame(
    [
        {"KPI": "MAE", "Where used": "Current price and forecast future price", "Why it matters": "Average dollar miss; easiest business interpretation", "Direction": "Lower is better", "Notebook output": "current_model_kpis, cohort_kpis"},
        {"KPI": "RMSE", "Where used": "Current price and forecast future price", "Why it matters": "Penalizes large misses and outlier errors", "Direction": "Lower is better", "Notebook output": "current_model_kpis, cohort_kpis"},
        {"KPI": "MAPE / WAPE", "Where used": "Relative price error", "Why it matters": "Compares misses across different price levels", "Direction": "Lower is better", "Notebook output": "current_model_kpis, cohort_kpis"},
        {"KPI": "RMSLE", "Where used": "Current price", "Why it matters": "Tracks multiplicative error on skewed positive prices", "Direction": "Lower is better", "Notebook output": "current_model_kpis"},
        {"KPI": "R2", "Where used": "Model comparison", "Why it matters": "Explained variance; useful but not sufficient alone", "Direction": "Higher is better", "Notebook output": "current_model_kpis, cohort_kpis"},
        {"KPI": "Skill vs naive", "Where used": "Cohort forecasts", "Why it matters": "Shows whether the model beats a no-change price forecast", "Direction": "Higher is better; must be above 0", "Notebook output": "cohort_kpis"},
        {"KPI": "Bias", "Where used": "Cohort forecasts", "Why it matters": "Flags systematic overpricing or underpricing", "Direction": "Closer to zero is better", "Notebook output": "cohort_kpis"},
        {"KPI": "Segment MAE", "Where used": "Price bands, high-value, make, year", "Why it matters": "Prevents average performance from hiding weak cohorts", "Direction": "Lower is better", "Notebook output": "segment tables"},
        {"KPI": "Top 30 weights", "Where used": "Readable/current and cohort models", "Why it matters": "Shows which engineered features the model relies on most", "Direction": "Review for plausibility and leakage risk", "Notebook output": "current_feature_weights, cohort_feature_importance"},
        {"KPI": "Forecast coverage", "Where used": "Cohort forecast output CSV", "Why it matters": "Confirms the forecast artifact spans the expected cohorts and months", "Direction": "Rows and max forecast month should match the run contract", "Notebook output": "forecast coverage table"},
    ]
)
metric_contract

,KPI,Where used,Why it matters,Direction,Notebook output
0,MAE,Current price and forecast future price,Average dollar miss; easiest business interpre...,Lower is better,"current_model_kpis, cohort_kpis"
1,RMSE,Current price and forecast future price,Penalizes large misses and outlier errors,Lower is better,"current_model_kpis, cohort_kpis"
2,MAPE / WAPE,Relative price error,Compares misses across different price levels,Lower is better,"current_model_kpis, cohort_kpis"
3,RMSLE,Current price,Tracks multiplicative error on skewed positive...,Lower is better,current_model_kpis
4,R2,Model comparison,Explained variance; useful but not sufficient ...,Higher is better,"current_model_kpis, cohort_kpis"
5,Skill vs naive,Cohort forecasts,Shows whether the model beats a no-change pric...,Higher is better; must be above 0,cohort_kpis
6,Bias,Cohort forecasts,Flags systematic overpricing or underpricing,Closer to zero is better,cohort_kpis
7,Segment MAE,"Price bands, high-value, make, year",Prevents average performance from hiding weak ...,Lower is better,segment tables
8,Top 30 weights,Readable/current and cohort models,Shows which engineered features the model reli...,Review for plausibility and leakage risk,"current_feature_weights, cohort_feature_import..."
9,Forecast coverage,Cohort forecast output CSV,Confirms the forecast artifact spans the expec...,Rows and max forecast month should match the r...,forecast coverage table


## Current-Price Model Results

The current-price report is generated by `ML/Price_ML_Models.py` and saved as `MODELS_OUTPUT/model_report.json`. The leaderboard below ranks models by MAE, with RMSE, MAPE, RMSLE, and R2 retained for model-quality context.


In [16]:
current_rows = []
fit_metadata = current_report.get("model_fit_metadata", {})
for model_name, metrics in current_report["models"].items():
    fit_info = fit_metadata.get(model_name, {})
    current_rows.append(
        {
            "model": model_name,
            "mae": metrics.get("mae"),
            "rmse": metrics.get("rmse"),
            "mape": metrics.get("mape"),
            "rmsle": metrics.get("rmsle"),
            "r2": metrics.get("r2"),
            "fit_rows": fit_info.get("final_fit_rows"),
            "fit_strategy": fit_info.get("final_fit_strategy"),
        }
    )

current_model_kpis = pd.DataFrame(current_rows).sort_values("mae")
current_model_kpis.style.format(
    {
        "mae": "${:,.0f}",
        "rmse": "${:,.0f}",
        "mape": "{:.1%}",
        "rmsle": "{:.3f}",
        "r2": "{:.3f}",
        "fit_rows": "{:,.0f}",
    },
    na_rep="",
)


,model,mae,rmse,mape,rmsle,r2,fit_rows,fit_strategy
4,Segmented_HighValue_LightGBM,"$1,941","$6,295",7.5%,0.125,0.937,"2,678,670",full_training_split
2,Advanced_LightGBM,"$2,048","$6,471",7.8%,0.127,0.934,"2,678,670",full_training_split
3,Tree_RandomForest,"$2,860","$10,968",11.4%,0.182,0.810,"300,000",representative_bounded_fit
0,Readable_Ridge,"$5,430","$94,755",13.3%,0.187,-13.168,"2,678,670",full_training_split
1,Readable_ElasticNet,"$5,449","$95,312",13.3%,0.187,-13.335,"2,678,670",full_training_split


In [17]:
fig, ax = plt.subplots(figsize=(10, 4.8))
plot_df = current_model_kpis.sort_values("mae", ascending=True)
ax.barh(plot_df["model"], plot_df["mae"], color="#2f6f8f")
ax.set_title("Current-price model MAE by model")
ax.set_xlabel("Mean absolute error ($)")
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.25)
for idx, value in enumerate(plot_df["mae"]):
    ax.text(value, idx, f"  ${value:,.0f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()


C:\Users\danie\AppData\Local\Temp\ipykernel_20936\109712694.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Feature Weights And Fit Footprint

The simple-model rows expose signed coefficients after preprocessing and scaling. Tree-family rows expose model-native feature importance, which is useful for ranking variables but should not be read as a causal effect.

In [18]:
def extract_weights_from_artifact(model_name: str, top_n: int = TOP_MODEL_WEIGHT_ROWS) -> list[dict]:
    path = OUTPUT_DIR / f"{model_name}.joblib"
    if not path.exists():
        return []
    model = joblib.load(path)
    if "preprocessor" not in model.named_steps or "model" not in model.named_steps:
        return []
    estimator = getattr(model.named_steps["model"], "regressor_", model.named_steps["model"])
    weight_type = "feature_importance"
    if hasattr(estimator, "coef_"):
        weights = np.ravel(estimator.coef_).astype(float)
        weight_type = "coefficient"
    elif hasattr(estimator, "feature_importances_"):
        weights = np.ravel(estimator.feature_importances_).astype(float)
    elif hasattr(estimator, "global_regressor_") and hasattr(estimator.global_regressor_, "feature_importances_"):
        weights = np.ravel(estimator.global_regressor_.feature_importances_).astype(float)
        weight_type = "global_feature_importance"
    else:
        return []
    try:
        feature_names = list(model.named_steps["preprocessor"].get_feature_names_out())
    except Exception:
        feature_names = [f"feature_{idx}" for idx in range(len(weights))]
    if len(feature_names) != len(weights):
        feature_names = [f"feature_{idx}" for idx in range(len(weights))]
    rows = [
        {
            "model": model_name,
            "feature": feature,
            "weight": float(weight),
            "abs_weight": abs(float(weight)),
            "weight_type": weight_type,
        }
        for feature, weight in zip(feature_names, weights)
    ]
    return sorted(rows, key=lambda row: row["abs_weight"], reverse=True)[:top_n]

feature_profile = current_report.get("feature_space_profile", {})
if feature_profile:
    display(
        pd.DataFrame(
            [
                {"metric": "raw pandas training frame", "value": f"{feature_profile.get('raw_pandas_training_frame_mb', np.nan):,.0f} MB"},
                {"metric": "profiled transformed shape", "value": str(tuple(feature_profile.get("transformed_shape_on_profile", [])))},
                {"metric": "transformed dtype", "value": feature_profile.get("transformed_dtype")},
                {"metric": "projected full transformed matrix", "value": f"{feature_profile.get('projected_full_training_matrix_memory_mb', np.nan) / 1024:,.2f} GB"},
            ]
        )
    )

weight_rows = []
saved_weights = current_report.get("model_feature_weights", {})
models_to_report = list(dict.fromkeys(["Readable_Ridge", "Readable_ElasticNet", best_model]))
for model_name in models_to_report:
    rows = saved_weights.get(model_name) or extract_weights_from_artifact(model_name)
    for rank, row in enumerate(rows[:TOP_MODEL_WEIGHT_ROWS], start=1):
        row = dict(row)
        row.setdefault("model", model_name)
        row["rank"] = rank
        weight_rows.append(row)

current_feature_weights = pd.DataFrame(weight_rows)
if current_feature_weights.empty:
    display(Markdown("No saved feature weights were found. Rerun `ML/Price_ML_Models.py` after the latest script update."))
else:
    display(Markdown(f"Showing the top {TOP_MODEL_WEIGHT_ROWS} weights/importances for each readable model and the selected best model."))
    display(
        current_feature_weights[["model", "rank", "feature", "weight", "abs_weight", "weight_type"]]
        .sort_values(["model", "rank"])
        .style.format({"weight": "{:.4g}", "abs_weight": "{:.4g}"})
    )

,metric,value
0,raw pandas training frame,"12,631 MB"
1,profiled transformed shape,"(200000, 335)"
2,transformed dtype,float32
3,projected full transformed matrix,3.34 GB


Showing the top 30 weights/importances for each readable model and the selected best model.

,model,rank,feature,weight,abs_weight,weight_type
30,Readable_ElasticNet,1,mileage_age_interaction,-0.4047,0.4047,coefficient
31,Readable_ElasticNet,2,target_encoded__title_normalized,0.1066,0.1066,coefficient
32,Readable_ElasticNet,3,vehicle_age_squared,0.1004,0.1004,coefficient
33,Readable_ElasticNet,4,nhtsa_DisplacementL,0.08552,0.08552,coefficient
34,Readable_ElasticNet,5,target_encoded__nhtsa_Model,0.08222,0.08222,coefficient
35,Readable_ElasticNet,6,vehicle_age,0.0593,0.0593,coefficient
36,Readable_ElasticNet,7,mileage_bucket_150k_plus,-0.05829,0.05829,coefficient
37,Readable_ElasticNet,8,nhtsa_EngineCylinders,0.05161,0.05161,coefficient
38,Readable_ElasticNet,9,target_encoded__nhtsa_EngineManufacturer,-0.04559,0.04559,coefficient
39,Readable_ElasticNet,10,nhtsa_DriveType_4WD/4-Wheel Drive/4x4,0.04095,0.04095,coefficient


## Segment Checks

The best model should be judged on the segments that matter operationally, not only on the global average. These tables use the segment metrics already saved in the current-price model report.


In [19]:
best_segment_metrics = current_report["models"][best_model].get("segment_metrics", {})

def segment_table(segment_name: str, min_rows: int = 1) -> pd.DataFrame:
    rows = []
    for segment_value, stats in best_segment_metrics.get(segment_name, {}).items():
        rows.append(
            {
                "segment": segment_value,
                "rows": stats.get("rows", 0),
                "mae": stats.get("mae", np.nan),
            }
        )
    if not rows:
        return pd.DataFrame(columns=["segment", "rows", "mae"])
    return (
        pd.DataFrame(rows)
        .query("rows >= @min_rows")
        .sort_values("mae", ascending=False)
        .reset_index(drop=True)
    )

price_band_check = segment_table("price_band")
high_value_check = segment_table("is_high_value")
make_check = segment_table("nhtsa_Make", min_rows=100).head(12)

display(Markdown("### Price Band MAE"))
display(price_band_check.style.format({"mae": "${:,.0f}"}))
display(Markdown("### High-Value Slice"))
display(high_value_check.style.format({"mae": "${:,.0f}"}))
display(Markdown("### Highest-MAE Makes With At Least 100 Test Rows"))
display(make_check.style.format({"mae": "${:,.0f}"}))


### Price Band MAE

,segment,rows,mae
0,150k_plus,2918,"$32,472"
1,100k_150k,3288,"$17,677"
2,50k_100k,57115,"$3,944"
3,25k_50k,289993,"$1,694"
4,under_25k,316354,"$1,360"


### High-Value Slice

,segment,rows,mae
0,high_value,2920,"$32,467"
1,everyday,666748,"$1,807"


### Highest-MAE Makes With At Least 100 Test Rows

,segment,rows,mae
0,FERRARI,630,"$44,273"
1,LAMBORGHINI,547,"$38,324"
2,MCLAREN,216,"$31,056"
3,PORSCHE,8616,"$8,784"
4,MASERATI,965,"$4,442"
5,JAGUAR,1346,"$3,286"
6,BMW,26884,"$2,751"
7,CADILLAC,10111,"$2,737"
8,DODGE,15345,"$2,423"
9,FORD,62551,"$2,377"


## Cohort Forecast Model Results

The time-series script trains cohort-level depreciation models by make, model, model year, and trim proxy. Direct horizon models require enough historical rows where the future target is already known. When direct long-horizon labels are too sparse, the script records a skip reason instead of failing and still writes one-year recursive projections from the shortest trained horizon model.


In [20]:
cohort_rows = []
for target, model in cohort_report.get("models", {}).items():
    metrics = model.get("metrics", {})
    cohort_rows.append(
        {
            "target": target,
            "horizon_months": horizon_months_from_target(target),
            "status": "skipped" if model.get("skipped") else "trained",
            "complete_rows": model.get("complete_rows"),
            "train_rows": model.get("train_rows"),
            "test_rows": model.get("test_rows"),
            "future_price_mae": metrics.get("future_price_mae"),
            "future_price_wape": metrics.get("future_price_wape"),
            "naive_mae": metrics.get("naive_no_change_future_price_mae"),
            "skill_vs_naive": metrics.get("future_price_mae_skill_vs_naive"),
            "beats_naive": metrics.get("future_price_mae_skill_vs_naive", np.nan) > 0,
            "depreciation_mae_pct_points": metrics.get("depreciation_mae_pct_points"),
            "bias_dollars": metrics.get("future_price_bias"),
            "depreciation_r2": metrics.get("depreciation_r2"),
            "skip_reason": model.get("skipped") or model.get("validation_warning"),
        }
    )

cohort_kpis = pd.DataFrame(cohort_rows).sort_values(["status", "skill_vs_naive"], ascending=[True, False])
if not cohort_summary.empty:
    best = cohort_summary.sort_values(["skill_vs_naive", "future_price_wape"], ascending=[False, True]).iloc[0]
    display(Markdown(f"**Best validated time-series horizon by skill vs naive:** `{best['target']}` ({int(best['horizon_months'])} month), skill {pct(best['skill_vs_naive'])}, WAPE {pct(best['future_price_wape'])}, MAE {money(best['future_price_mae'])}."))
cohort_kpis.style.format(
    {
        "future_price_mae": "${:,.0f}",
        "future_price_wape": "{:.1%}",
        "naive_mae": "${:,.0f}",
        "skill_vs_naive": "{:.1%}",
        "depreciation_mae_pct_points": "{:.2f}",
        "bias_dollars": "${:,.0f}",
        "depreciation_r2": "{:.3f}",
    },
    na_rep="",
)

**Best validated time-series horizon by skill vs naive:** `target_depreciation_pct_1m` (1 month), skill -6.9%, WAPE 5.1%, MAE $1,968.

,target,horizon_months,status,complete_rows,train_rows,test_rows,future_price_mae,future_price_wape,naive_mae,skill_vs_naive,beats_naive,depreciation_mae_pct_points,bias_dollars,depreciation_r2,skip_reason
0,target_depreciation_pct_1m,1,trained,50496,40663,7734,"$1,968",5.1%,"$1,842",-6.9%,False,6.70,$-759,0.159,


## Time-Series Feature Importance

These rows use the best validated cohort horizon from the time-series report. They show model-native LightGBM importance at the cohort-month grain.


In [21]:
def cohort_feature_importance_for_target(target: str, top_n: int = TOP_MODEL_WEIGHT_ROWS) -> pd.DataFrame:
    model_report = cohort_report.get("models", {}).get(target, {})
    rows = model_report.get("feature_importance") or []
    if not rows and model_report.get("artifact"):
        path = OUTPUT_DIR / model_report["artifact"]
        if path.exists():
            model = joblib.load(path)
            estimator = model.named_steps.get("model")
            if hasattr(estimator, "feature_importances_"):
                weights = np.ravel(estimator.feature_importances_).astype(float)
                try:
                    feature_names = list(model.named_steps["preprocessor"].get_feature_names_out())
                except Exception:
                    feature_names = [f"feature_{idx}" for idx in range(len(weights))]
                if len(feature_names) != len(weights):
                    feature_names = [f"feature_{idx}" for idx in range(len(weights))]
                rows = [
                    {"feature": feature, "importance": float(weight), "abs_weight": abs(float(weight)), "weight_type": "feature_importance"}
                    for feature, weight in zip(feature_names, weights)
                ]
    if not rows:
        return pd.DataFrame()
    out = pd.DataFrame(rows).sort_values("abs_weight", ascending=False).head(top_n).reset_index(drop=True)
    out.insert(0, "rank", np.arange(1, len(out) + 1))
    return out

if best_cohort is None:
    display(Markdown("No trained cohort model metrics were found."))
else:
    best_target = best_cohort["target"]
    cohort_feature_importance = cohort_feature_importance_for_target(best_target)
    if cohort_feature_importance.empty:
        display(Markdown("No feature importance was found for the best cohort model. Rerun `ML/Time_Series_Price.py` after the latest script update."))
    else:
        display(Markdown(f"### Top {TOP_MODEL_WEIGHT_ROWS} drivers for `{best_target}`"))
        display(cohort_feature_importance[["rank", "feature", "importance", "abs_weight", "weight_type"]].style.format({"importance": "{:,.0f}", "abs_weight": "{:,.0f}"}))

### Top 30 drivers for `target_depreciation_pct_1m`

,rank,feature,importance,abs_weight,weight_type
0,1,price_index_vs_cohort_first,"1,861","1,861",feature_importance
1,2,lag_price_index_1,"1,551","1,551",feature_importance
2,3,median_price,"1,392","1,392",feature_importance
3,4,rolling_depreciation_pct_3m,"1,258","1,258",feature_importance
4,5,avg_vehicle_age_months,"1,051","1,051",feature_importance
5,6,rolling_avg_mileage_3m,973,973,feature_importance
6,7,avg_miles_per_year,960,960,feature_importance
7,8,lag_median_price_2,949,949,feature_importance
8,9,median_mileage,877,877,feature_importance
9,10,market_monthly_volume,873,873,feature_importance


In [22]:
if forecast_df.empty:
    display(Markdown("No forecast CSV was found. Run `ML/Time_Series_Price.py` to generate `cohort_future_forecasts.csv`."))
else:
    observed_col = "observed_month_start" if "observed_month_start" in forecast_df.columns else "observed_week_start"
    horizon_col = "forecast_month" if "forecast_month" in forecast_df.columns else "forecast_horizon_days"
    horizon_label = "max forecast month" if horizon_col == "forecast_month" else "max horizon days"
    coverage = pd.DataFrame(
        [
            {"metric": "forecast rows", "value": f"{len(forecast_df):,}"},
            {"metric": "cohorts", "value": f"{forecast_df[['make', 'model', 'model_year', 'trim_proxy']].drop_duplicates().shape[0]:,}"},
            {"metric": "last observed period", "value": str(forecast_df[observed_col].max().date())},
            {"metric": "max forecast date", "value": str(forecast_df["forecast_date"].max().date())},
            {"metric": horizon_label, "value": f"{forecast_df[horizon_col].max():,.0f}"},
        ]
    )
    display(coverage)
    display(
        forecast_df.groupby(["forecast_method", horizon_col], as_index=False)
        .size()
        .rename(columns={"size": "rows"})
        .sort_values([horizon_col, "forecast_method"])
        .tail(20)
    )

,metric,value
0,forecast rows,"2,380,080"
1,cohorts,"39,668"
2,last observed period,2026-07-01
3,max forecast date,2031-07-01
4,max forecast month,60


,forecast_method,forecast_month,rows
40,recursive_1m_model,41,39668
41,recursive_1m_model,42,39668
42,recursive_1m_model,43,39668
43,recursive_1m_model,44,39668
44,recursive_1m_model,45,39668
45,recursive_1m_model,46,39668
46,recursive_1m_model,47,39668
47,recursive_1m_model,48,39668
48,recursive_1m_model,49,39668
49,recursive_1m_model,50,39668


In [23]:
if not forecast_df.empty:
    horizon_col = "forecast_month" if "forecast_month" in forecast_df.columns else "forecast_horizon_days"
    target_horizon = 12 if horizon_col == "forecast_month" and forecast_df[horizon_col].max() >= 12 else forecast_df[horizon_col].max()
    if pd.notna(target_horizon):
        horizon_frame = forecast_df[forecast_df[horizon_col].eq(target_horizon)].copy()
        horizon_frame["predicted_depreciation_pct"] = pd.to_numeric(horizon_frame["predicted_depreciation_pct"], errors="coerce")
        horizon_label = f"{int(target_horizon)}-month" if horizon_col == "forecast_month" else f"{int(target_horizon)}-day"
        fig, ax = plt.subplots(figsize=(10, 4.8))
        ax.hist(horizon_frame["predicted_depreciation_pct"].dropna(), bins=40, color="#8a6f2a", edgecolor="white")
        ax.set_title(f"{horizon_label.capitalize()} cohort forecast distribution")
        ax.set_xlabel("Predicted price change from latest observed median")
        ax.set_ylabel("Cohorts")
        ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
        ax.grid(axis="y", alpha=0.25)
        plt.tight_layout()
        plt.show()

        horizon_summary = horizon_frame["predicted_depreciation_pct"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).to_frame("predicted_depreciation_pct")
        display(horizon_summary.style.format("{:.1%}"))

C:\Users\danie\AppData\Local\Temp\ipykernel_20936\2686998839.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,predicted_depreciation_pct
count,3966800.0%
mean,8.4%
std,41.6%
min,-82.3%
10%,-19.6%
25%,-10.1%
50%,-1.5%
75%,13.5%
90%,43.2%
max,1099.4%


## Takeaways

- The current-price model should be monitored primarily with MAE, RMSE, MAPE/RMSLE, and segment MAE. The selected saved run currently favors the segmented LightGBM model, but high-value and exotic segments remain materially harder than everyday-price vehicles.
- The readable Ridge and ElasticNet models now expose the top 30 signed coefficients so the strongest feature weights can be reviewed for plausibility, leakage risk, and alignment with the research questions.
- The cohort forecast model reports baseline skill, WAPE, and bias in addition to depreciation error. In the latest saved output, the one-month model has negative skill versus the no-change baseline, so recursive forecasts should be treated as directional scenarios rather than a proven improvement.
- The cohort forecast output is monthly and currently extends to the report's `max_forecast_date`. Use the forecast coverage table to confirm row counts, cohort counts, and forecast-month span after each model run.
- The next model-quality improvement should be adding more historical price depth or lowering cohort sparsity, then rerunning the notebook to see whether the direct monthly depreciation model beats the naive no-change baseline and whether longer direct horizons become trainable.


## Sources

- scikit-learn, [Metrics and scoring: quantifying the quality of predictions](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics)
- Hyndman and Athanasopoulos, [Forecasting: Principles and Practice, Evaluating point forecast accuracy](https://otexts.com/fpp3/accuracy.html)
- Hyndman and Koehler, [Another look at measures of forecast accuracy](https://robjhyndman.com/publications/another-look-at-measures-of-forecast-accuracy/)
- Pal et al., [How much is my car worth? A methodology for predicting used cars prices using Random Forest](https://arxiv.org/abs/1711.06970)
- Madhusudhanan et al., [ProbSAINT: Probabilistic Tabular Regression for Used Car Pricing](https://arxiv.org/abs/2403.03812)
- Montero-Manso and Hyndman, [Principles and Algorithms for Forecasting Groups of Time Series: Locality and Globality](https://arxiv.org/abs/2008.00444)
- Hewamalage, Bergmeir, and Bandara, [Global Models for Time Series Forecasting: A Simulation Study](https://arxiv.org/abs/2012.12485)
- Manheim, [Used Vehicle Value Index Summary Methodology](https://site.manheim.com/wp-content/uploads/sites/2/2024/02/Used-Vehicle-Summary-Methodology.pdf)
